# 13 Observer | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: powiadamianie wielu obiektow
2. 📢 Struktura: Subject i Observer
3. 🐍 Implementacja z typowaniem (`Callable`, `Protocol`)
4. 📦 Zdarzenia jako obiekty (`@dataclass Event`)
5. 🔗 Zastosowania i logging jako Observer

## 1. 🔹 Problem: powiadamianie wielu obiektow

Observer (Obserwator, Listener, Event-Driven) to wzorzec
behawioralny pozwalajacy zdefiniowac mechanizm subskrypcji
do powiadamiania wielu obiektow o zdarzeniach.

Problem bez Observer:
- Obiekt A zmienia stan i musi powiadomic B, C, D
- A zna B, C, D bezposrednio - silna zaleznosc
- Dodanie nowego obserwatora E wymaga zmiany A
- Naruszenie OCP i SRP

Observer rozwiazuje:
- Subject (Podmiot) nie zna konkretnych obserwatorow
- Obserwatorzy subskrybuja/odsubskrybuja dynamicznie
- Luzsne sprzezenie (loose coupling): Subject wie tylko o interfejsie Observer

Nazewnictwo w roznych kontekstach:
- Subject -> Publisher / EventEmitter / Observable
- Observer -> Subscriber / Listener / Handler
- update() -> on_event() / handle() / notify()

> 💡 Observer to podstawa programowania zdarzeniowego (event-driven).
> Jest wszyudziie: GUI, Node.js EventEmitter, Python logging.

In [ ]:
# Problem bez Observer: silne sprzezenie
class ShoppingCart:
    def __init__(self):
        self._items = []
        # Znamy konkretne klasy - silna zaleznosc!
        self._email_service = None  # EmailService()
        self._analytics = None      # Analytics()
        self._inventory = None      # Inventory()

    def add_item(self, item: dict) -> None:
        self._items.append(item)
        # Musimy recznie wywolywac kazdy system
        if self._email_service:
            self._email_service.send_cart_update(item)
        if self._analytics:
            self._analytics.track('item_added', item)
        if self._inventory:
            self._inventory.reserve(item['id'])
        # Dodanie nowego systemu = zmiana ShoppingCart!

problems = [
    'ShoppingCart zna EmailService, Analytics, Inventory',
    'Dodanie nowego systemu wymaga zmiany add_item()',
    'Trudno testowac (mock calej zaleznosci)',
    'Naruszone SRP: ShoppingCart zarzadza powiadomieniami',
]
for p in problems:
    print(f'- {p}')

---

### 🐍 Cwiczenia - problem

1. Rozszerz `ShoppingCart` o 4. system `LoyaltyPoints`.
   Policz ile linii kodu musisz zmienic.
2. Napisz `WeatherStation` bez Observer powiadamiajaca
   `PhoneDisplay`, `WebDisplay`, `Logger` przy zmianie temp.
3. *(Trudniejsze)* Zidentyfikuj problem z polaczeniami N:M
   gdy 5 obiektow musi sie nawzajem powiadamiac (bez Observer).

In [ ]:
# Cwiczenie 1: dodanie LoyaltyPoints bez Observer
class ShoppingCartV2(ShoppingCart):
    def __init__(self):
        super().__init__()
        self._loyalty = None  # LoyaltyPoints()

    def add_item(self, item: dict) -> None:
        super().add_item(item)
        if self._loyalty:
            self._loyalty.add_points(item.get('price', 0))

print('Zmiany w kodzie:')
print('1. Dodac self._loyalty w __init__')
print('2. Dodac if self._loyalty w add_item')
print('Z Observer: 0 zmian w ShoppingCart!')

In [ ]:
# Cwiczenie 2: WeatherStation bez Observer
class PhoneDisplay:
    def show(self, temp: float): print(f'Phone: {temp}°C')

class WebDisplay:
    def show(self, temp: float): print(f'Web: {temp}°C')

class WeatherStation:
    def __init__(self):
        self._temp = 20.0
        self._phone = PhoneDisplay()
        self._web = WebDisplay()

    def set_temperature(self, temp: float) -> None:
        self._temp = temp
        # Recznie powiadamiamy - silne sprzezenie
        self._phone.show(temp)
        self._web.show(temp)
        # Dodanie LogDisplay wymaga zmiany tutaj!

ws = WeatherStation()
ws.set_temperature(25.0)
print('Problem: WeatherStation zna PhoneDisplay i WebDisplay')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: problem N:M bez Observer
n = 5
bezposrednie_polaczenia = n * (n - 1)  # kazdy z kazdym
z_mediator = n  # kazdy zna tylko mediatora

print(f'Obiektow: {n}')
print(f'Polaczen bez Mediatora/Observera: {bezposrednie_polaczenia}')
print(f'Polaczen z Observerem: {n} (kazdy subskrybuje)')
print(f'Redukcja: {bezposrednie_polaczenia / z_mediator:.0f}x')

## 2. 🔹 Struktura: Subject i Observer

Klasyczna implementacja wzorca Observer:

**Subject (Podmiot)**:
- `subscribe(observer)` - dodaje obserwatora
- `unsubscribe(observer)` - usuwa obserwatora
- `notify()` - powiadamia wszystkich obserwatorow

**Observer (Obserwator)**:
- `update(event, data)` - wywoływane przez Subject

Dwa modele powiadamiania:
1. **Push**: Subject wysyla dane do Observer w update()
   - Observer nie musi wywolywac Subject
   - Dane moga byc nadmiarowe

2. **Pull**: Observer pyta Subject o stan po update()
   - Observer dostaje tylko powiadomienie
   - Sam pobiera interesujace go dane

Rozrzednia lista obserwatorow (weak references):
- Unikamy wycieku pamieci gdy Observer jest usuniety
- Subject nie powinien podtrzymywac zycia Observera

In [ ]:
from abc import ABC, abstractmethod

# Observer interface
class Observer(ABC):
    @abstractmethod
    def update(self, event: str, data: dict) -> None: ...

# Subject mixin
class Subject:
    def __init__(self) -> None:
        self._observers: list[Observer] = []

    def subscribe(self, observer: Observer) -> None:
        if observer not in self._observers:
            self._observers.append(observer)

    def unsubscribe(self, observer: Observer) -> None:
        self._observers.remove(observer)

    def notify(self, event: str, data: dict) -> None:
        for obs in list(self._observers):  # kopia - bezpieczna iteracja
            obs.update(event, data)

# Konkretny Subject
class ShoppingCart(Subject):
    def __init__(self):
        super().__init__()
        self._items: list[dict] = []

    def add_item(self, item: dict) -> None:
        self._items.append(item)
        self.notify('item_added', {'item': item, 'total': len(self._items)})

    def checkout(self) -> None:
        total = sum(i['price'] for i in self._items)
        self.notify('checkout', {'items': self._items, 'total': total})
        self._items.clear()

# Konkretni Observatorzy
class EmailNotifier(Observer):
    def __init__(self, email: str): self._email = email
    def update(self, event: str, data: dict) -> None:
        if event == 'checkout':
            print(f'Email to {self._email}: Order confirmed, total={data["total"]}')

class Analytics(Observer):
    def update(self, event: str, data: dict) -> None:
        print(f'Analytics.track({event!r}): {data}')

class InventoryManager(Observer):
    def update(self, event: str, data: dict) -> None:
        if event == 'item_added':
            print(f'Inventory.reserve: {data["item"]["name"]}')

cart = ShoppingCart()
cart.subscribe(EmailNotifier('alice@x.com'))
cart.subscribe(Analytics())
cart.subscribe(InventoryManager())

cart.add_item({'name': 'Widget', 'price': 9.99})
cart.add_item({'name': 'Gadget', 'price': 29.99})
cart.checkout()

print('\n--- After checkout, unsubscribe Analytics ---')
cart.subscribe(EmailNotifier('alice@x.com'))
cart.unsubscribe(cart._observers[1])  # usunieta Analytics nie jest juz notyfikowana

---

### 🐍 Cwiczenia - Subject / Observer

1. Rozszerz `ShoppingCart` o `LoyaltyPoints(Observer)` dodajacy
   punkty (1 punkt za 1 PLN) przy checkout.
2. Napisz `WeatherStation(Subject)` i obserwatorow
   `PhoneDisplay`, `WebDisplay`, `CSVLogger`.
3. *(Trudniejsze)* Dodaj priorytet obserwatorow:
   `subscribe(observer, priority: int)` - obserwatorzy
   z wyzszym priorytetem sa powiadamiani pierwsi.

In [ ]:
# Cwiczenie 1: LoyaltyPoints
class LoyaltyPoints(Observer):
    def __init__(self): self._points = 0
    def update(self, event: str, data: dict) -> None:
        ...
    @property
    def points(self): return self._points

cart2 = ShoppingCart()
loyalty = LoyaltyPoints()
cart2.subscribe(loyalty)
cart2.add_item({'name': 'Widget', 'price': 50.0})
cart2.checkout()
print(f'Points earned: {loyalty.points}')

In [ ]:
# Cwiczenie 2: WeatherStation
class WeatherStation(Subject):
    def __init__(self):
        super().__init__()
        self._temp = 20.0
        self._humidity = 60.0

    def update_weather(self, temp: float, humidity: float) -> None:
        self._temp = temp
        self._humidity = humidity
        self.notify('weather_updated', {'temp': temp, 'humidity': humidity})

class PhoneDisplay(Observer):
    def update(self, event: str, data: dict) -> None:
        print(f'Phone: {data["temp"]}°C, {data["humidity"]}%')

class CSVLogger(Observer):
    def __init__(self): self._records = []
    def update(self, event: str, data: dict) -> None:
        self._records.append(data)
    def get_records(self): return self._records

station = WeatherStation()
logger2 = CSVLogger()
station.subscribe(PhoneDisplay())
station.subscribe(logger2)
station.update_weather(22.0, 55.0)
station.update_weather(25.0, 70.0)
print(f'Logged {len(logger2.get_records())} records')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: priorytet obserwatorow
class PrioritySubject:
    def __init__(self):
        self._observers: list[tuple[int, Observer]] = []

    def subscribe(self, observer: Observer, priority: int = 0) -> None:
        # hint: wstaw w posortowanej pozycji (wyzszy priorytet = wczesniej)
        ...

    def unsubscribe(self, observer: Observer) -> None:
        self._observers = [(p, o) for p, o in self._observers if o is not observer]

    def notify(self, event: str, data: dict) -> None:
        for _, obs in self._observers:
            obs.update(event, data)

class LogObserver(Observer):
    def __init__(self, name: str, priority: int):
        self.name = name; self.priority = priority
    def update(self, event: str, data: dict) -> None:
        print(f'[P{self.priority}] {self.name}: {event}')

ps = PrioritySubject()
ps.subscribe(LogObserver('Low', 1), priority=1)
ps.subscribe(LogObserver('High', 10), priority=10)
ps.subscribe(LogObserver('Mid', 5), priority=5)
ps.notify('test', {})
# Oczekiwany output: High (10), Mid (5), Low (1)

## 3. 🔹 Implementacja z typowaniem (`Callable`, `Protocol`)

W Python Observer moze byc:
1. **Klasa z interfejsem**: `class Observer(ABC): def update()`
2. **Callable**: zwykla funkcja lub lambda
3. **Protocol**: duck typing z typowaniem statycznym

Callable jako Observer:
- Prostsze - nie trzeba tworzyc klasy
- `bus.on('event', lambda data: ...)`
- Python `logging` uzywa handler jako callable

`typing.Protocol`:
- Definiuje interfejs bez dziedziczenia
- Klasa spełniajaca interfejs nie musi po nim dziedziczyc
- Lepsze wsparcie dla type checkerow (mypy, pyright)

```python
class Observer(Protocol):
    def update(self, event: str, data: dict) -> None: ...
```

Callbacki z rejestracja:
- Dekorator `@bus.on('event')` - czytelna rejestracja
- `functools.wraps` - zachowanie metadanych

In [ ]:
from typing import Callable, Protocol
import functools

# Podejscie 1: Callable jako Observer
class EventEmitter:
    def __init__(self) -> None:
        self._handlers: dict[str, list[Callable]] = {}

    def on(self, event: str) -> Callable:
        """Dekorator: @emitter.on('event')."""
        def decorator(func: Callable) -> Callable:
            self._handlers.setdefault(event, []).append(func)
            return func
        return decorator

    def emit(self, event: str, **data) -> None:
        for handler in self._handlers.get(event, []):
            handler(**data)

emitter = EventEmitter()

@emitter.on('user.login')
def send_welcome_email(user: str, **kw) -> None:
    print(f'Welcome email to: {user}')

@emitter.on('user.login')
def log_login(user: str, **kw) -> None:
    print(f'Audit: {user} logged in')

@emitter.on('order.placed')
def notify_warehouse(order_id: int, items: list, **kw) -> None:
    print(f'Warehouse: prepare order #{order_id}: {items}')

emitter.emit('user.login', user='alice')
emitter.emit('order.placed', order_id=42, items=['Widget x2'])


# Podejscie 2: Protocol dla typowania
class ObserverProtocol(Protocol):
    def update(self, event: str, data: dict) -> None: ...

class TypedSubject:
    def __init__(self) -> None:
        self._observers: list[ObserverProtocol] = []

    def subscribe(self, observer: ObserverProtocol) -> None:
        self._observers.append(observer)

    def notify(self, event: str, data: dict) -> None:
        for obs in self._observers:
            obs.update(event, data)

# Klasa NIE musi dziedziczyc po ObserverProtocol
class AuditLog:
    def update(self, event: str, data: dict) -> None:  # spełnia Protocol!
        print(f'AUDIT: {event} -> {data}')

ts = TypedSubject()
ts.subscribe(AuditLog())  # dziala bez dziedziczenia
ts.notify('user.created', {'name': 'Bob'})

---

### 🐍 Cwiczenia - Callable / Protocol

1. Napisz `EventBus` z metodami `on(event, handler)` i `emit(event, **data)`
   gdzie handler to `Callable(**data) -> None`.
2. Uzyj `@emitter.on('click')` jako dekoratora do rejestracji
   trzech roznych handlerow klikniecia.
3. *(Trudniejsze)* Napisz `Once(handler)` - handler ktory wykonuje
   sie tylko raz, potem automatycznie odpsubskrybuje.

In [ ]:
# Cwiczenie 1: EventBus z Callable
class EventBus:
    def __init__(self):
        self._handlers: dict[str, list[Callable]] = {}

    def on(self, event: str, handler: Callable) -> None:
        ...

    def off(self, event: str, handler: Callable) -> None:
        ...

    def emit(self, event: str, **data) -> None:
        ...

bus = EventBus()
bus.on('message', lambda text, **kw: print(f'Chat: {text}'))
bus.on('message', lambda text, **kw: print(f'Log: {text}'))
bus.emit('message', text='Hello!')

In [ ]:
# Cwiczenie 2: dekorator @emitter.on
app_emitter = EventEmitter()

@app_emitter.on('click')
def handle_click_log(x, y, **kw):
    print(f'[Log] click at ({x}, {y})')

@app_emitter.on('click')
def handle_click_sound(x, y, **kw):
    print(f'[Sound] click sound')

@app_emitter.on('click')
def handle_click_tooltip(x, y, **kw):
    print(f'[Tooltip] show tooltip at ({x}, {y})')

app_emitter.emit('click', x=100, y=200)

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Once handler
def once(bus: EventBus, event: str, handler: Callable) -> None:
    # hint: wrapper wywoluje handler, potem odsubskrybowuje siebie
    def wrapper(**data):
        ...
    bus.on(event, wrapper)

bus2 = EventBus()
once(bus2, 'connect', lambda **kw: print('Connected! (only once)'))
bus2.on('connect', lambda **kw: print('Also connected (always)'))
bus2.emit('connect')
bus2.emit('connect')  # once handler nie powinien sie wykonac

## 4. 🔹 Zdarzenia jako obiekty (`@dataclass Event`)

Zamiast przekazywac event_name: str i data: dict,
mozna tworzyc typowane obiekty zdarzen.

Zalety typowanych zdarzen:
- Statyczne typowanie (IDE wie co jest w zdarzeniu)
- Walidacja w __init__ (przez dataclass lub pydantic)
- Latwiejsze logowanie i serializacja
- Mozliwos dziedziczenia zdarzen

```python
@dataclass
class Event:
    timestamp: float = field(default_factory=time.time)

@dataclass
class UserLoginEvent(Event):
    user_id: int
    ip_address: str
```

Dyspatcher oparty na typach:
- `bus.on(UserLoginEvent, handler)` - subskrypcja przez typ
- `bus.emit(UserLoginEvent(...))` - emisja obiektu
- Handler dostaje typowany obiekt zdarzenia

> 💡 To podejscie jest wzorcem 'Domain Events' z DDD
> (Domain-Driven Design). Zdarzenia sa czescia modelu domeny.

In [ ]:
from dataclasses import dataclass, field
import time
from typing import Callable, Type, TypeVar

T = TypeVar('T')

@dataclass
class DomainEvent:
    timestamp: float = field(default_factory=time.time)

@dataclass
class UserCreatedEvent(DomainEvent):
    user_id: int = 0
    email: str = ''

@dataclass
class OrderPlacedEvent(DomainEvent):
    order_id: int = 0
    user_id: int = 0
    total: float = 0.0
    items: list = field(default_factory=list)

@dataclass
class PaymentProcessedEvent(DomainEvent):
    order_id: int = 0
    amount: float = 0.0
    status: str = 'pending'

class DomainEventBus:
    def __init__(self) -> None:
        self._handlers: dict[type, list[Callable]] = {}

    def on(self, event_type: type, handler: Callable) -> None:
        self._handlers.setdefault(event_type, []).append(handler)

    def emit(self, event: DomainEvent) -> None:
        handlers = self._handlers.get(type(event), [])
        for h in handlers:
            h(event)

bus = DomainEventBus()

# Subskrypcje - handler wie dokladnie jaki typ dostaje
bus.on(UserCreatedEvent, lambda e: print(f'Send welcome email to {e.email}'))
bus.on(UserCreatedEvent, lambda e: print(f'Create user profile for #{e.user_id}'))
bus.on(OrderPlacedEvent, lambda e: print(f'Order #{e.order_id}: {len(e.items)} items, total {e.total}'))
bus.on(OrderPlacedEvent, lambda e: print(f'Notify warehouse for order #{e.order_id}'))
bus.on(PaymentProcessedEvent, lambda e: print(f'Payment {e.status} for order #{e.order_id}: {e.amount}'))

# Emisja
bus.emit(UserCreatedEvent(user_id=1, email='alice@x.com'))
bus.emit(OrderPlacedEvent(order_id=42, user_id=1, total=99.99, items=['Widget', 'Gadget']))
bus.emit(PaymentProcessedEvent(order_id=42, amount=99.99, status='succeeded'))

---

### 🐍 Cwiczenia - zdarzenia typowane

1. Zdefiniuj `TemperatureChangedEvent(prev, current, unit)` i
   `ThresholdExceededEvent(threshold, value)`. Napisz handlery.
2. Napisz hierarchie zdarzen dla systemu plikow:
   `FileEvent`, `FileCreatedEvent`, `FileDeletedEvent`, `FileModifiedEvent`.
3. *(Trudniejsze)* Napisz `EventMiddleware` - middleware dla EventBus
   ktore loguje kazde zdarzenie przed przekazaniem do handlerow.

In [ ]:
# Cwiczenie 1: zdarzenia temperatury
@dataclass
class TemperatureChangedEvent(DomainEvent):
    prev: float = 0.0
    current: float = 0.0
    unit: str = 'C'

@dataclass
class ThresholdExceededEvent(DomainEvent):
    threshold: float = 0.0
    value: float = 0.0
    direction: str = 'above'

temp_bus = DomainEventBus()
temp_bus.on(TemperatureChangedEvent, lambda e: print(f'Temp: {e.prev} -> {e.current}{e.unit}'))
temp_bus.on(ThresholdExceededEvent, lambda e: print(f'Threshold {e.direction}: {e.value} (limit: {e.threshold})'))

temp_bus.emit(TemperatureChangedEvent(prev=20.0, current=25.0))
temp_bus.emit(ThresholdExceededEvent(threshold=30.0, value=35.0, direction='above'))

In [ ]:
# Cwiczenie 2: hierarchia FileEvent
@dataclass
class FileEvent(DomainEvent):
    path: str = ''

@dataclass
class FileCreatedEvent(FileEvent):
    size: int = 0

@dataclass
class FileDeletedEvent(FileEvent):
    pass

@dataclass
class FileModifiedEvent(FileEvent):
    old_size: int = 0
    new_size: int = 0

fs_bus = DomainEventBus()
fs_bus.on(FileCreatedEvent, lambda e: print(f'Created: {e.path} ({e.size} B)'))
fs_bus.on(FileDeletedEvent, lambda e: print(f'Deleted: {e.path}'))
fs_bus.on(FileModifiedEvent, lambda e: print(f'Modified: {e.path} ({e.old_size} -> {e.new_size} B)'))

fs_bus.emit(FileCreatedEvent(path='/tmp/a.txt', size=1024))
fs_bus.emit(FileModifiedEvent(path='/tmp/a.txt', old_size=1024, new_size=2048))
fs_bus.emit(FileDeletedEvent(path='/tmp/a.txt'))

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: EventMiddleware
class EventMiddleware:
    def __init__(self, bus: DomainEventBus):
        self._bus = bus
        self._emit_count = 0

    def emit(self, event: DomainEvent) -> None:
        self._emit_count += 1
        event_type = type(event).__name__
        print(f'[Middleware] emit #{self._emit_count}: {event_type}')
        self._bus.emit(event)
        print(f'[Middleware] done: {event_type}')

mw = EventMiddleware(DomainEventBus())
mw._bus.on(UserCreatedEvent, lambda e: print(f'Handler: user {e.user_id}'))
mw.emit(UserCreatedEvent(user_id=99, email='test@x.com'))
print(f'Total emitted: {mw._emit_count}')

## 5. 🔹 Zastosowania i logging jako Observer

Python `logging` module to klasyczny przyklad Observer:
- `Logger` (Subject): generuje LogRecord
- `Handler` (Observer): StreamHandler, FileHandler, SMTPHandler
- `Logger.addHandler(handler)` - subskrypcja
- `logger.info(msg)` - emit zdarzenia
- `Filter` - warunek powiadamiania

Inne zastosowania Observer w Pythonie:
- `tkinter`: `widget.bind('event', handler)` - event loop
- Django signals: `post_save.connect(handler, sender=Model)`
- SQLAlchemy events: `@event.listens_for(Model, 'before_insert')`
- asyncio: `loop.add_reader(fd, callback)`

Reaktywne programowanie (Reactive Programming):
- RxPY: `Observable.subscribe(observer)`
- Observer to fundament RxJS/RxPY

> 💡 Observer jest tak powszechny ze w wielu jezykach
> wbudowany jest w rdzen (C# events, Java EventListener,
> JavaScript EventEmitter).

In [ ]:
import logging

# logging.Handler to Observer
# logging.Logger to Subject

# Konfiguracja - jak subskrypcja Observer
logger = logging.getLogger('myapp')
logger.setLevel(logging.DEBUG)

# Handler = ConcreteObserver
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))

# addHandler = subscribe
logger.addHandler(console_handler)

# Drugi observer - inny poziom
debug_handler = logging.StreamHandler()
debug_handler.setLevel(logging.DEBUG)
debug_handler.setFormatter(logging.Formatter('DEBUG: %(message)s'))
logger.addHandler(debug_handler)

# emit zdarzenia
logger.debug('detail')     # tylko debug_handler
logger.info('started')     # oba handlery
logger.error('crash!')

print('\nLogging module = Observer Pattern!')
print('Logger = Subject')
print('Handler = ConcreteObserver')
print('addHandler = subscribe')
print('logger.info() = notify')

---

### 🐍 Cwiczenia - zastosowania

1. Skonfiguruj logger z dwoma handlerami: ConsoleHandler (INFO+)
   i MemoryHandler (DEBUG+). Po 5 logach wyswietl zawartosc pamieci.
2. Napisz `ReactiveProperty(value)` - property ktora powiadamia
   obserwatorow gdy jej wartosc sie zmieni (jak Knockout.js).
3. *(Trudniejsze)* Napisz mini RxPY: `Observable` z metodami
   `subscribe(observer)` i `map(func)`, `filter(pred)`.

In [ ]:
# Cwiczenie 1: logging z MemoryHandler
class MemoryLogHandler(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []
    def emit(self, record) -> None:
        self.records.append(self.format(record))

mem_logger = logging.getLogger('mem_test')
mem_logger.setLevel(logging.DEBUG)
mem_logger.addHandler(logging.StreamHandler())  # console INFO+
mem_handler = MemoryLogHandler()
mem_handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
mem_logger.addHandler(mem_handler)

for i in range(5):
    if i % 2 == 0: mem_logger.debug(f'debug {i}')
    else: mem_logger.info(f'info {i}')

print(f'\nMemory records ({len(mem_handler.records)}):')
for r in mem_handler.records: print(f'  {r}')

In [ ]:
# Cwiczenie 2: ReactiveProperty
class ReactiveProperty:
    def __init__(self, value):
        self._value = value
        self._observers: list[Callable] = []

    def subscribe(self, observer: Callable) -> None:
        self._observers.append(observer)

    @property
    def value(self): return self._value

    @value.setter
    def value(self, new_val) -> None:
        old = self._value
        self._value = new_val
        for obs in self._observers:
            obs(old, new_val)

price = ReactiveProperty(100.0)
price.subscribe(lambda old, new: print(f'Price: {old} -> {new}'))
price.subscribe(lambda old, new: print(f'Change: {((new-old)/old)*100:.1f}%'))
price.value = 110.0
price.value = 95.0

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: mini RxPY
class Observable:
    def __init__(self, producer):
        self._producer = producer
        self._ops = []

    def subscribe(self, on_next: Callable, on_error: Callable = None, on_complete: Callable = None):
        def safe_next(v):
            try: on_next(v)
            except Exception as e:
                if on_error: on_error(e)

        # Aplikuj transformacje
        def emit(v):
            result = v
            for op in self._ops:
                result = op(result)
                if result is None: return  # filter zwrocil None
            safe_next(result)

        self._producer(emit)
        if on_complete: on_complete()

    def map(self, func: Callable) -> 'Observable':
        new_obs = Observable(self._producer)
        new_obs._ops = self._ops + [func]
        return new_obs

    def filter(self, pred: Callable) -> 'Observable':
        def filter_op(v):
            return v if pred(v) else None
        new_obs = Observable(self._producer)
        new_obs._ops = self._ops + [filter_op]
        return new_obs

# Tworzymy Observable z listy
from_list = lambda items: Observable(lambda emit: [emit(x) for x in items])

result = []
(from_list([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
    .filter(lambda x: x % 2 == 0)
    .map(lambda x: x * x)
    .subscribe(result.append))
print('Squares of even:', result)